In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ==========================================
# 1. DEFINE THE PROBLEM & THE OBJECTIVE FUNCTION
# ==========================================
def objective_function(x, y):
    return -(x**2 + y**2) + 4*x + 4*y + 10

x_min, x_max = -5, 5
y_min, y_max = -5, 5

# ==========================================
# 2. INITIALIZE HYPERPARAMETERS
# ==========================================
num_particles = 30
max_iterations = 40

w = 0.5
c1 = 1.5
c2 = 1.5

# ==========================================
# 3. INITIALIZE PARTICLES
# ==========================================
positions = np.random.uniform(low=[x_min, y_min], high=[x_max, y_max], size=(num_particles, 2))
velocities = np.random.uniform(low=-1, high=1, size=(num_particles, 2))

fitness_scores = objective_function(positions[:, 0], positions[:, 1])

pbest_positions = np.copy(positions)
pbest_fitness = np.copy(fitness_scores)

best_particle_idx = np.argmax(pbest_fitness)
gbest_position = np.copy(pbest_positions[best_particle_idx])
gbest_fitness_value = pbest_fitness[best_particle_idx] # Track as a scalar value

gbest_history = [gbest_position.copy()]
position_history = [positions.copy()]

# ==========================================
# 4. RUN THE PSO OPTIMIZATION LOOP
# ==========================================
for iteration in range(max_iterations):
    r1 = np.random.rand(num_particles, 2)
    r2 = np.random.rand(num_particles, 2)

    # Update Velocities & Positions
    velocities = (w * velocities) + \
                 (c1 * r1 * (pbest_positions - positions)) + \
                 (c2 * r2 * (gbest_position - positions))
    positions = positions + velocities

    # Boundary Clipping
    positions[:, 0] = np.clip(positions[:, 0], x_min, x_max)
    positions[:, 1] = np.clip(positions[:, 1], y_min, y_max)

    position_history.append(positions.copy())

    # Evaluate New Fitness
    current_fitness = objective_function(positions[:, 0], positions[:, 1])

    # Update Personal Best (pbest) safely loop-by-loop or index matched
    for i in range(num_particles):
        if current_fitness[i] > pbest_fitness[i]:
            pbest_fitness[i] = current_fitness[i]
            pbest_positions[i] = positions[i]

    # Update Global Best (gbest) safely using individual elements
    best_current_idx = np.argmax(pbest_fitness)
    if pbest_fitness[best_current_idx] > gbest_fitness_value:
        gbest_fitness_value = pbest_fitness[best_current_idx]
        gbest_position = np.copy(pbest_positions[best_current_idx])

    gbest_history.append(gbest_position.copy())

print(f"Optimization Finished Successfully!")
print(f"Discovered Best Position: x = {gbest_position[0]:.4f}, y = {gbest_position[1]:.4f}")
print(f"Maximum Fitness Score Achieved: {gbest_fitness_value:.4f}")

# ==========================================
# 5. CREATING INTERACTIVE ANIMATION FOR JUPYTER
# ==========================================
fig, ax = plt.subplots(figsize=(8, 6))

x_grid = np.linspace(x_min, x_max, 100)
y_grid = np.linspace(y_min, y_max, 100)
X, Y = np.meshgrid(x_grid, y_grid)
Z = objective_function(X, Y)
contour = ax.contourf(X, Y, Z, levels=20, cmap='viridis')
fig.colorbar(contour, label='Fitness Value')

particles_scatter = ax.scatter([], [], color='red', edgecolor='black', s=50, label='Swarm Particles')
gbest_scatter = ax.scatter([], [], color='gold', marker='*', s=250, edgecolor='black', label='Global Leader (gbest)')

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_xlabel('X Coordinate')
ax.set_ylabel('Y Coordinate')
ax.legend(loc='upper left')

def init():
    particles_scatter.set_offsets(np.empty((0, 2)))
    gbest_scatter.set_offsets(np.empty((0, 2)))
    return particles_scatter, gbest_scatter

def update(frame):
    ax.set_title(f"PSO Iteration {frame}/{max_iterations}")
    current_pos = position_history[frame]
    particles_scatter.set_offsets(current_pos)
    current_gbest = gbest_history[frame]
    gbest_scatter.set_offsets([current_gbest])
    return particles_scatter, gbest_scatter

anim = FuncAnimation(fig, update, frames=max_iterations, init_func=init, blit=True, interval=150)
plt.close(fig)

HTML(anim.to_html5_video())


Optimization Finished Successfully!
Discovered Best Position: x = 2.0000, y = 2.0000
Maximum Fitness Score Achieved: 18.0000
